# Experiment 11 — Rollout fidelity

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Simulate complete negotiation trajectories using the fitted
linear policy and a buyer that mirrors the original data-generation rule
($B_t \sim U[0, S_{t-1}]$).  Compare the synthetic trajectories' final-price
distribution and per-round mean ask against the real data.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression

def fit_rollout_model(policy):
    feat, mdl, alpha = SELECTED[policy]
    meta, X, y, cols = nu.build_design_matrix(turns, policy, feat)
    scaler = StandardScaler().fit(X)
    y_logit = nu.logit_clip(y)
    if mdl == 'OLS':
        reg = LinearRegression().fit(scaler.transform(X), y_logit)
    else:
        a = float(alpha) if np.isfinite(alpha) else 1.0
        reg = Ridge(alpha=a).fit(scaler.transform(X), y_logit)
    return scaler, reg, cols, feat

def features_row_from_dict(d, cols):
    '''Build a 1xN feature row from a dict, matching cols order.'''
    row = np.array([[d.get(c, 0.0) for c in cols]], dtype=float)
    return row

def simulate_run(policy, v, M, T=16, rng=None, s_init_norm=0.9):
    '''Roll out one full negotiation. Returns arrays of S, B on the raw-dollar scale.'''
    rng = rng or np.random.default_rng()
    scaler, reg, cols, feat = ROLLOUT_MODELS[policy]
    S = np.zeros(T); B = np.zeros(T)
    # Seed: start seller at a high normalized ask, buyer uniform[0, S0]
    S[0] = v + s_init_norm * (M - v)
    B[0] = rng.uniform(0, S[0])
    S[1] = S[0] * 0.95   # small concession to get rolling
    B[1] = rng.uniform(0, S[1])
    for t in range(2, T):
        B[t] = rng.uniform(0, S[t-1])
        # Build feature dict in normalized coords
        Bn    = (B[t] - v) / (M - v)
        Bln   = (B[t-1] - v) / (M - v)
        Sln   = (S[t-1] - v) / (M - v)
        Bln2  = (B[t-2] - v) / (M - v)
        Sln2  = (S[t-2] - v) / (M - v)
        d = {'B_norm': Bn, 'S_lag1': Sln, 'B_lag1': Bln,
             'S_lag2': Sln2, 'B_lag2': Bln2,
             'A': Sln - Bn, 'A_lag1': Sln2 - Bln,
             'B_norm_pos': max(Bn, 0), 'B_norm_neg': max(-Bn, 0),
             'B_lag1_pos': max(Bln, 0), 'B_lag1_neg': max(-Bln, 0),
             'dB': Bn - Bln, 'dS_lag1': Sln - Sln2,
             'dB_pos': max(Bn-Bln, 0), 'dB_neg': max(-(Bn-Bln), 0),
             'dS_lag1_pos': max(Sln-Sln2, 0), 'dS_lag1_neg': max(-(Sln-Sln2), 0)}
        Xrow = features_row_from_dict(d, cols)
        yhat_logit = reg.predict(scaler.transform(Xrow))[0]
        S_norm_pred = float(nu.sigmoid(yhat_logit))
        # Clip to [0, 1] then unnormalize
        S[t] = v + np.clip(S_norm_pred, 0, 1) * (M - v)
        # Optional: break if agreement
        if S[t] <= 1.01 * B[t]:
            S[t+1:] = S[t]; B[t+1:] = B[t]
            break
    return S, B

ROLLOUT_MODELS = {p: fit_rollout_model(p) for p in POLICIES}

# Simulate N runs per (policy, vehicle)
fleet = turns.groupby('vehicle').agg(v=('v','first'), M=('M','first'))
N_SIMS = 30

sim_rows = []
rng = np.random.default_rng(42)
for p in POLICIES:
    for veh, row in fleet.iterrows():
        for rid in range(N_SIMS):
            S, B = simulate_run(p, row['v'], row['M'], T=16, rng=rng)
            for t in range(16):
                sim_rows.append({'policy': p, 'vehicle': veh, 'run_sim': rid,
                                 'round': t+1, 'S_sim': S[t], 'B_sim': B[t],
                                 'v': row['v'], 'M': row['M']})
sim = pd.DataFrame(sim_rows)
print(f'Simulated {len(sim):,} rows.')
sim.head()

### Mean seller ask per round: real vs simulated

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for i, p in enumerate(POLICIES):
    ax = axes.flatten()[i]
    real = (turns[(turns['policy'] == p) & (~turns['parse_fail'])]
             .groupby('round')['S_norm'].agg(['mean', 'std']))
    ssim = sim[sim['policy'] == p].assign(S_norm=lambda d: (d['S_sim']-d['v'])/(d['M']-d['v']))
    simm = ssim.groupby('round')['S_norm'].agg(['mean', 'std'])
    ax.plot(real.index, real['mean'], 'o-', color='tab:blue', label='real')
    ax.fill_between(real.index, real['mean']-real['std'], real['mean']+real['std'], color='tab:blue', alpha=0.2)
    ax.plot(simm.index, simm['mean'], 's--', color='tab:red', label='sim')
    ax.fill_between(simm.index, simm['mean']-simm['std'], simm['mean']+simm['std'], color='tab:red', alpha=0.2)
    ax.set_title(p); ax.set_xlabel('round'); ax.set_ylim(0, 1)
    if i == 0: ax.set_ylabel(r'$\tilde S_t$  (mean $\pm$ SD)')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figE11_rollout_mean.png', dpi=140, bbox_inches='tight')
plt.show()